# 3_analise.ipynb — Camada Gold, perguntas de negócio e gráficos

**Projeto:** Viagens a Serviço (Portal da Transparência)
**Fluxo de camadas:** Raw → Silver → **Gold**


---

### O que este notebook faz?

Este notebook representa a **Fase 3** do pipeline ETL. Ele parte das tabelas **Silver** — já limpas e tipadas pelo script `2_transformar.py` — e dá dois passos:

1. **Constrói a camada Gold**, com tabelas e *views* agregadas que resumem os dados para análise.
2. **Responde as 7 perguntas de negócio** propostas no projeto.

Toda pergunta segue o mesmo roteiro, o que mantém a análise padronizada e fácil de acompanhar:


- Pergunta  →  Consulta SQL  →  Tabela  →  Gráfico  →  Conclusão


---


### Regra de análise adotada

Todas as perguntas consideram **apenas viagens efetivamente realizadas**, ou seja, registros com:

```sql
situacao = 'Realizada'
```

**Por quê?**
Viagens canceladas, não iniciadas ou ainda em aberto **não representam gasto nem deslocamento real**. Incluí-las distorceria médias, somas e rankings. Filtrar por `'Realizada'` garante que os números reflitam apenas o que de fato aconteceu.

---

### Onde cada pergunta é respondida

As perguntas estão divididas conforme a camada usada para respondê-las: algumas saem direto da **Silver**, outras exigem agregações da **Gold** (com `JOIN` e `GROUP BY`).

| Camada | Tipo de consulta | Perguntas |
|---|---|---|
| **Silver** | Consulta direta | 1. Órgãos com maior custo <br> 2. Destinos com maior custo médio <br> 3. Viagem de maior duração |
| **Gold** | Tabela e view com `JOIN` + `GROUP BY` | 4. Tipo de pagamento com maior valor médio <br> 5. Meio de transporte mais usado <br> 6. UF de destino mais frequente <br> 7. Órgão que mais pagou |



### Antes de executar

Para que este notebook funcione corretamente, **execute os scripts na ordem**:

1. `0_criar_banco.sql` — cria o banco e as tabelas das camadas Raw e Silver;
2. `1_extrair.py` — baixa os CSVs e carrega a camada Raw;
3. `2_transformar.py` — limpa, tipa e carrega a camada Silver.

Além disso, confira se o arquivo **`.env`** está preenchido com a sua senha do PostgreSQL.

⚠️ Sem esses pré-requisitos atendidos, o notebook não conseguirá ler a Silver e, portanto, não conseguirá gerar a Gold nem responder às perguntas.

---

### Preparação: bibliotecas e conexão

Antes de qualquer consulta ou gráfico, o notebook precisa preparar três coisas: **as bibliotecas**, **a conexão com o banco** e **as funções auxiliares** que vão ser usadas em todas as etapas seguintes.



#### Bibliotecas utilizadas

| Biblioteca | Para que serve neste notebook |
|---|---|
| `pandas` | Organiza o resultado das consultas SQL em tabelas (DataFrames). |
| `matplotlib` | Desenha os gráficos a partir dessas tabelas. |
| `banco.conectar()` | Abre a conexão com o banco `transparencia`, usando a senha definida no arquivo `.env`. |

**Por que usar `banco.conectar()`?**

Toda a configuração do PostgreSQL (host, porta, usuário, senha e nome do banco) fica centralizada no `.env`. Assim, o notebook não precisa repetir credenciais — basta reaproveitar a função já criada no módulo `banco.py`.

In [ ]:
# teste de conexão com o banco de dados
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

import matplotlib.pyplot as plt
import pandas as pd

from banco import conectar


# mostra os números das tabelas com 2 casas decimais
pd.set_option("display.float_format", "{:.2f}".format)


def consultar(sql):
    conexao = conectar()
    tabela = pd.read_sql(sql, conexao)
    conexao.close()
    return tabela


def executar(sql):
    """Executa comandos SQL sem retorno (CREATE, DROP, INSERT...)."""
    conexao = conectar()              # abre a conexão
    conexao.autocommit = True         # confirma cada comando automaticamente
    cursor = conexao.cursor()         # cria o cursor
    try:
        cursor.execute(sql)           # executa o SQL
    finally:
        cursor.close()                # fecha o cursor
        conexao.close()               # fecha a conexão


def formatar_numero(valor, casas=2):
    texto = f"{valor:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


teste = consultar("SELECT COUNT(*) AS viagens FROM silver_viagem")
print("Conexão OK. Viagens na camada Silver:", teste["viagens"][0])

---
# Respondendo as peguntas de negócio:

### 1. Quais são os 5 órgãos com maior custo total?

In [ ]:
# Busca os 5 órgãos com maior custo total, considerando só viagens realizadas
sql_top_orgaos = """
    SELECT nome_orgao_superior AS orgao,     
           SUM(valor_total)    AS custo_total -- soma o custo de todas as viagens do órgão
    FROM silver_viagem
    WHERE situacao = 'Realizada'             -- ignora viagens canceladas/não realizadas
    GROUP BY nome_orgao_superior             -- agrupa por órgão
    ORDER BY custo_total DESC                -- ordena do maior para o menor
    LIMIT 5                                  -- só os 5 primeiros
"""

# Executa a consulta e devolve como tabela do Pandas
tabela = consultar(sql_top_orgaos)

# Cria uma coluna extra em milhões, para o gráfico ficar mais legível
tabela["custo_milhoes"] = tabela["custo_total"] / 1_000_000

# Mostra a tabela formatada na tela
print(f"Os 5 órgãos com maior custo total são:")
display(tabela)


# Gráfico de barras horizontais
# Cria a figura e o eixo, com tamanho confortável para leitura
fig, ax = plt.subplots(figsize=(10, 4))

# Desenha as barras: órgão no eixo Y, custo em milhões no eixo X
ax.barh(
    tabela["orgao"],
    tabela["custo_milhoes"],
    color="#821DBD",         
    label="Custo total das viagens",
)

# Inverte o eixo Y para o maior valor ficar no topo
ax.invert_yaxis()

# Define títulos dos eixos e do gráfico
ax.set_title("Os 5 órgãos com maior custo total em viagens a serviço (jan-jun/2025)\n")
ax.set_xlabel("Custo total (R$ milhões)")
ax.set_ylabel("Órgão superior")

# Coloca a legenda fora do gráfico, para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

# Ajusta o layout para não cortar nada e exibe o gráfico
plt.tight_layout()
plt.show()


# Diferença entre o primeiro e o segundo colocado da lista
dif = tabela["custo_total"].iloc[0] - tabela["custo_total"].iloc[1]
print(f"É possível identificar uma diferença significativa de R$ {(dif / 1_000_000):.2f} milhões de reais entre o primeiro e segundo colocados da lista de órgãos.")

---
### 2. Os 3 destinos com maior custo médio por viagem?

In [ ]:
# ===========================================================================
# Pergunta 2: quais destinos finais têm o maior custo médio por viagem?
# Estratégia: criar uma VIEW no PostgreSQL que já devolve o destino final
# (última cidade do roteiro) e reutilizá-la em qualquer consulta futura.
# ===========================================================================


# ---------------------------------------------------------------------------
# Passo 1 — Cria (ou substitui) a view com a coluna destino_final
# ---------------------------------------------------------------------------

# SQL que cria a view no banco. Se ela já existir, é substituída.
# A ideia: pegar tudo o que vem DEPOIS da última vírgula da coluna destinos.

# TRIM - Remove espaços em branco do começo e do fim do texto.
# SPLIT_PART(texto, separador, posição) 

sql_criar_view = """
    CREATE OR REPLACE VIEW vw_viagem_destino_final AS
    SELECT
        id_viagem,
        situacao,
        valor_total,
        TRIM(
            SPLIT_PART(
                RTRIM(destinos, ', '),  -- limpa vírgulas e espaços do final
                ',',
                -1
            )
        ) AS destino_final  -- quebra a coluna "destinos" separando por "," e pega a última posição
        FROM silver_viagem;
"""
    
# Executa a criação da view direto no banco
executar(sql_criar_view)

print("View vw_viagem_destino_final criada (ou atualizada).")


In [ ]:
# ---------------------------------------------------------------------------
# Passo 2 — Consulta a view, já com o destino final pronto
# ---------------------------------------------------------------------------

# Filtra viagens realizadas e traz o que interessa.
sql_destinos = """
    SELECT destino_final,
           valor_total
    FROM vw_viagem_destino_final
    WHERE situacao = 'Realizada'
"""

# Executa a consulta — já vem com a coluna destino_final separada
tabela = consultar(sql_destinos)

# Mostra uma amostra para conferir se a separação ficou correta
display(tabela.head())

In [ ]:
# ---------------------------------------------------------------------------
# Passo 3 — Agrupa por destino final e calcula viagens + custo médio
# ---------------------------------------------------------------------------

# Agrupa pela coluna destino_final, conta viagens e calcula a média do custo
resumo = (
    tabela
    .groupby("destino_final")
    .agg(
        viagens=("valor_total", "count"),       # quantas viagens terminaram nesse destino
        custo_medio=("valor_total", "mean"),    # média do custo por viagem
    )
    .reset_index()
)

# Arredonda o custo médio para 2 casas decimais
resumo["custo_medio"] = resumo["custo_medio"].round(2)

# Mostra quantos destinos finais únicos existem antes do filtro
print(f"Destinos finais únicos encontrados: {len(resumo)}")
#display(resumo.head())

In [ ]:
# ---------------------------------------------------------------------------
# Passo 4 — Filtra destinos finais com 100 viagens ou mais
# ---------------------------------------------------------------------------

# Só entram destinos finais com pelo menos 100 viagens
resumo = resumo[resumo["viagens"] >= 100]

# Ordena do maior custo médio para o menor e pega os 3 primeiros
top3 = resumo.sort_values("custo_medio", ascending=False).head(3).reset_index(drop=True)

# Mostra o resultado final
display(top3)

In [ ]:

# ---------------------------------------------------------------------------
# Passo 5 — Gráfico de barras verticais
# ---------------------------------------------------------------------------

# Cria a figura e o eixo, com proporção mais alta para caber os nomes embaixo
fig, ax = plt.subplots(figsize=(8, 6))

# Desenha as barras: destino final no eixo X, custo médio no eixo Y
ax.bar(
    top3["destino_final"],
    top3["custo_medio"],
    color="#D68E08",                    
    label="Custo médio por viagem",
)

# Títulos do gráfico e dos eixos
ax.set_title("Os 3 destinos finais com maior custo médio por viagem (mínimo de 100 viagens)\n")
ax.set_xlabel("Destino final")
ax.set_ylabel("Custo médio por viagem (R$)")

# Gira os rótulos do eixo X em 30º, para nomes longos não se sobreporem
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Legenda fora do gráfico, para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

# Ajusta o layout e exibe
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# Passo 6 — Conclusão em texto
# ---------------------------------------------------------------------------

# Pega o primeiro colocado do ranking
lider = top3.iloc[0]

# Imprime a conclusão com o nome do destino final já extraído pela view
print(
    f"Conclusão: \n{lider['destino_final']} é o destino final com o maior custo médio por viagem: R$ {formatar_numero(lider['custo_medio'])}, "
    f"em {lider['viagens']} viagens. "
    f"\nAqui só entram destinos finais com 100 viagens ou mais, para a média não ficar destorcida."
)

---
### 3. A viagem de maior duração e seu custo total?

In [ ]:
# Busca as 5 viagens realizadas com maior duração
tabela = consultar("""
    SELECT id_viagem,
           nome_orgao_superior AS orgao,
           data_inicio,
           data_fim,
           duracao_dias,
           valor_total
    FROM silver_viagem
    WHERE situacao = 'Realizada' AND duracao_dias IS NOT NULL
    ORDER BY duracao_dias DESC, valor_total DESC
    LIMIT 5
""")

# Mostra os dados encontrados
display(tabela)

# Cria o gráfico de barras horizontais
fig, ax = plt.subplots(figsize=(9, 3.8))

# Compara a duração das 5 viagens
ax.barh(
        tabela["id_viagem"], 
        tabela["duracao_dias"], 
        color="#3C792D", 
        label="Duração da viagem"
        )

# Coloca a viagem mais longa no topo
ax.invert_yaxis()

ax.set_title("As 5 viagens de maior duração")
ax.set_xlabel("Duração (dias)")
ax.set_ylabel("Identificador da viagem")

# Deixa a legenda fora do gráfico para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

# Pega a primeira linha, que corresponde à viagem mais longa
maior = tabela.iloc[0]

# Exibe um resumo com o período e o custo da viagem
print(f"Conclusão: \nA viagem {maior['id_viagem']} ({maior['orgao']}) é a mais longa, com {maior['duracao_dias']} dias "
      f"(de {maior['data_inicio']} a {maior['data_fim']}). \nCusto total registrado: R$ {formatar_numero(maior['valor_total'])}."
      f"\nPode ser um erro de digitação, pois a viagem durou mais de um ano.")

---
## Camada Gold

Criando a camada Gold para responder as próximas perguntas de negócio.

In [ ]:
from banco import conectar

# Consulta o total pago por órgão e tipo de pagamento
CONSULTA_PAGAMENTO = """
    SELECT p.nome_orgao_pagador,
           p.tipo_pagamento,
           COUNT(*) AS qtd_pagamentos,
           SUM(p.valor) AS total_pago
    FROM silver_pagamento p
    JOIN silver_viagem v ON v.id_viagem = p.id_viagem
    WHERE v.situacao = 'Realizada'
    GROUP BY p.nome_orgao_pagador, p.tipo_pagamento
"""

# Conta os trechos por meio de transporte e estado de destino
CONSULTA_TRECHO = """
    SELECT t.meio_transporte,
           t.destino_uf,
           COUNT(*) AS qtd_trechos
    FROM silver_trecho t
    JOIN silver_viagem v ON v.id_viagem = t.id_viagem
    WHERE v.situacao = 'Realizada'
    GROUP BY t.meio_transporte, t.destino_uf
"""

# Abre a conexão com o banco de dados
conexao = conectar()
try:
    cursor = conexao.cursor()

    # Cria a tabela de resumo dos pagamentos
    cursor.execute("DROP TABLE IF EXISTS gold_pagamento_resumo")
    cursor.execute("CREATE TABLE gold_pagamento_resumo AS " + CONSULTA_PAGAMENTO)

    # Cria uma view com a mesma consulta de pagamentos
    cursor.execute("CREATE OR REPLACE VIEW vw_gold_pagamento_resumo AS " + CONSULTA_PAGAMENTO)

    # Cria a tabela de resumo dos trechos
    cursor.execute("DROP TABLE IF EXISTS gold_trecho_resumo")
    cursor.execute("CREATE TABLE gold_trecho_resumo AS " + CONSULTA_TRECHO)

    # Cria uma view com a mesma consulta de trechos
    cursor.execute("CREATE OR REPLACE VIEW vw_gold_trecho_resumo AS " + CONSULTA_TRECHO)

    # Salva as alterações no banco
    conexao.commit()
    print("Camada Gold criada: 2 tabelas e 2 views.")

except Exception as erro:
    # Desfaz as alterações se acontecer algum erro
    conexao.rollback()
    print("ERRO ao criar a camada Gold:", type(erro).__name__, erro)

finally:
    # Fecha a conexão com o banco
    conexao.close()

---
### 4. Qual o tipo de pagamento com maior valor médio? 

In [ ]:
tabela = consultar("""
    SELECT tipo_pagamento,
	   ROUND(SUM(total_pago), 2)                         AS valor_total,
       SUM(qtd_pagamentos)::INT                          AS qtd_pagamentos,       
       ROUND(SUM(total_pago) / SUM(qtd_pagamentos), 2)   AS valor_medio
    FROM gold_pagamento_resumo
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC;
    """)
display(tabela)

In [ ]:
# Monta o rótulo de cada fatia: nome do tipo em uma linha, valor em outra.
rotulos = [
    f"{tipo}\nR$ {formatar_numero(valor)}"
    for tipo, valor in zip(tabela["tipo_pagamento"], tabela["valor_medio"])
]

fig, ax = plt.subplots(figsize=(8, 5))

# Cria o gráfico de pizza
ax.pie(
    tabela["valor_medio"],
    labels=rotulos,
    autopct="%1.1f%%",
    startangle=90,
    colors=plt.cm.Blues(range(30, 180, 30)),
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    labeldistance=1.15,
    pctdistance=0.75,          # aproxima o percentual do centro
    textprops={"fontsize": 10},
)

ax.set_title("Valor médio por tipo de pagamento")
plt.tight_layout()
plt.show()

lider = tabela.iloc[0]
print(f"Conclusão: \nO tipo '{lider['tipo_pagamento']}' tem o maior valor médio, R$ {formatar_numero(lider['valor_medio'])} "
      f"por pagamento, em {formatar_numero(lider['qtd_pagamentos'], 0)} pagamentos.")

---
### 5. Qual o meio de transporte mais usado nos trechos? 


In [ ]:
# Soma a quantidade de trechos por meio de transporte
tabela = consultar("""
    SELECT meio_transporte,
           SUM(qtd_trechos)::INT AS trechos
    FROM gold_trecho_resumo
    GROUP BY meio_transporte
    ORDER BY trechos DESC
""")

# Mostra os dados da consulta
display(tabela)

# Separa os 4 primeiros e soma os demais em "Outros"
if len(tabela) > 4:
    principais = tabela.iloc[:4]
    outros = tabela.iloc[4:]["trechos"].sum()

    tabela = pd.concat([
        principais,
        pd.DataFrame([{"meio_transporte": "Outros", "trechos": outros}])
    ], ignore_index=True)

# Mostra a tabela resumida
# display(tabela)

# Cria o gráfico de barras horizontais
fig, ax = plt.subplots(figsize=(9, 4.2))

# Compara a quantidade de trechos de cada transporte
ax.barh(tabela["meio_transporte"], tabela["trechos"], color="#D451CE", label="Quantidade de trechos")

# Coloca o transporte mais usado no topo
ax.invert_yaxis()

ax.set_title("Meios de transporte mais usados nos trechos das viagens realizadas")
ax.set_xlabel("Quantidade de trechos")
ax.set_ylabel("Meio de transporte")

# Deixa a legenda fora do gráfico para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

# Pega o meio de transporte que aparece em primeiro lugar
lider = tabela.iloc[0]

# Soma todos os trechos para calcular a porcentagem
total = tabela["trechos"].sum()

# Exibe o transporte mais usado e sua participação no total
print(f"Conclusão: \n{lider['meio_transporte']} é o meio mais usado, com {formatar_numero(lider['trechos'], 0)} trechos "
      f"({formatar_numero(lider['trechos'] / total * 100, 1)}% do total de {formatar_numero(total, 0)}).")

---
### 6. Qual UF de destino aparece em mais trechos?

In [ ]:
# Busca as 5 UFs que aparecem com mais trechos de viagem
tabela = consultar("""
    SELECT destino_uf AS uf_de_destino,
           SUM(qtd_trechos)::INT AS trechos
    FROM gold_trecho_resumo
    GROUP BY destino_uf
    ORDER BY trechos DESC
    LIMIT 5
""")

# Mostra os dados encontrados
display(tabela)

# Cria o gráfico de barras horizontais
fig, ax = plt.subplots(figsize=(9, 4.5))

# Compara a quantidade de trechos por UF de destino
ax.barh(tabela["uf_de_destino"], tabela["trechos"], color="#E6D81D", label="Quantidade de trechos")

# Coloca a UF com mais trechos no topo
ax.invert_yaxis()

ax.set_title("As 5 UFs de destino com mais trechos")
ax.set_xlabel("Quantidade de trechos")
ax.set_ylabel("UF de destino")

# Deixa a legenda fora do gráfico para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

# Pega a UF que aparece em primeiro lugar
lider = tabela.iloc[0]

# Mostra a UF mais frequente e a quantidade de trechos
print(f"Conclusão: \n{lider['uf_de_destino']} é a UF de destino mais frequente, com {formatar_numero(lider['trechos'], 0)} trechos.")

---
### 7. Qual órgão pagou mais no total?

In [ ]:
# Busca os 5 órgãos que pagaram mais
tabela = consultar("""
    SELECT nome_orgao_pagador AS orgao_pagador,
           SUM(total_pago) AS total_pago
    FROM gold_pagamento_resumo
    GROUP BY nome_orgao_pagador
    ORDER BY total_pago DESC
    LIMIT 5
""")

# Converte os valores para milhões de reais, para facilitar a leitura
tabela["total_milhoes"] = tabela["total_pago"] / 1_000_000

# Mostra os dados encontrados
display(tabela)

# Cria o gráfico de barras horizontais
fig, ax = plt.subplots(figsize=(10, 4))

# Compara o total pago por cada órgão
ax.barh(tabela["orgao_pagador"], tabela["total_milhoes"], color="#04B3F8", label="Total pago")

# Coloca o órgão que mais pagou no topo
ax.invert_yaxis()

ax.set_title("Os 5 órgãos que mais pagaram em viagens a serviço")
ax.set_xlabel("Total pago (R$ milhões)")
ax.set_ylabel("Órgão pagador")

# Deixa a legenda fora do gráfico para não cobrir as barras
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

# Pega o órgão que teve o maior total de pagamentos
lider = tabela.iloc[0]

# Mostra o órgão que mais pagou e o valor total em milhões
print(f"Conclusão: \nO {lider['orgao_pagador']} foi quem mais pagou, R$ {formatar_numero(lider['total_milhoes'], 1)} milhões.")

## Decisões e limitações da análise

Durante o desenvolvimento, precisei tomar algumas decisões que não estavam definidas no enunciado. Registro aqui o que escolhi e por quê, para que o resultado seja reprodutível e transparente.

- **Filtro base: só viagens realizadas**\
Todas as perguntas consideram apenas `situacao = 'Realizada'`. Viagens canceladas ou não realizadas não representam deslocamento nem gasto efetivo, então ficam de fora para não distorcer médias e rankings.


- **Cálculo do custo total**\
O enunciado não define como somar os valores, então adotei:
    ```
    custo_total = diárias + passagens + outros gastos - devolução
    ```
    A devolução entra como subtração porque representa valor que voltou aos cofres públicos, ou seja, não é gasto real. A regra está aplicada no `2_transformar.py`, na coluna `valor_total`.


- **Cálculo da duração**\
Adotei `(data_fim - data_inicio) + 1` dia. O `+1` existe porque o dia de início e o dia de fim contam como dias de viagem. Sem ele, uma viagem de um único dia apareceria com duração zero.


- **Destinos (Pergunta 2)**\
A coluna `destinos` é texto livre e costuma trazer várias cidades separadas por vírgula. Extraí apenas a última cidade do roteiro, que é o destino final de fato. Também defini um piso de 100 viagens por destino, para evitar que cidades visitadas poucas vezes distorçam a média.


- **Média por tipo de pagamento (Pergunta 4)**\
Calculei como total pago dividido pela quantidade de pagamentos, e não como média das médias. Essa escolha evita que um tipo com poucos registros de valor alto fique diluído no resultado.


- **Meio de transporte (Pergunta 5)**\
A base traz uma categoria chamada "Inválido". Mantive como veio da fonte, porque representa o dado real publicado pelo Portal, mesmo que o nome não ajude a interpretar o resultado.


- **Órgão pagador (Pergunta 7)**\
Existe um pagador identificado apenas como "Sigiloso", que aparece entre os maiores valores e não permite saber qual órgão pagou. Optei por manter o registro, sinalizando a limitação, pois removê-lo esconderia parte do gasto real.
